# Praktikum 09 – Agenten und MCP-Server
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Agenten von Chatbots unterscheiden, das ReAct-Pattern implementieren, Tool Use mit Ollama nutzen und einen minimalen MCP-Server erstellen.

**Zielprodukt nach 90 Minuten:**
1. Ein ReAct-Agent mit Reasoning-Loop.
2. Mindestens zwei Tools (Rechner, Wissensabfrage).
3. Ein funktionierender MCP-Server mit zwei Endpunkten.

In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und führe die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Agent vs. Chatbot (15 min)
Ein Chatbot antwortet auf Eingaben, ein Agent plant und führt Aktionen aus. Wir zeigen den Unterschied konkret.

**Pflichtschritte:**
- Definiere, was einen Agenten ausmacht.
- Zeige ein Beispiel, das nur ein Chatbot kann.
- Zeige ein Beispiel, das nur ein Agent kann.

**Soll-Ergebnis:** Eine klare Abgrenzung in 3 bis 5 Sätzen.

In [ ]:
# Vergleich: Chatbot vs. Agent

CHATBOT_TASK = "Erkläre mir RAG in einem Satz."
AGENT_TASK = "Berechne 23 * 17, suche dann nach dem Ergebnis in meiner Wissensbasis und schreibe eine Zusammenfassung."


def chat_response(prompt):
    """Einfache Chatbot-Antwort ohne Werkzeuge."""
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 100},
        keep_alive="10m",
    )
    return response["message"]["content"].strip()


print("=== Chatbot-Beispiel ===")
print(f"Aufgabe: {CHATBOT_TASK}")
chat_result = chat_response(CHATBOT_TASK)
print(f"Antwort: {chat_result}")

print("\n=== Agent-Beispiel (erfordert Werkzeuge) ===")
print(f"Aufgabe: {AGENT_TASK}")
print("Ein Chatbot kann diese Aufgabe nicht selbstständig lösen, da er:")
print("1. Nicht rechnen kann (kein Calculator-Tool)")
print("2. Nicht in einer Wissensbasis suchen kann (kein Search-Tool)")
print("3. Nicht mehrere Schritte koordinieren kann (kein Planungsmodul)")

### Definition: Agent vs. Chatbot

**Chatbot:**
- Reagiert auf Benutzereingaben
- Hat keinen Zustand oder Plan
- Nutzt keine externen Werkzeuge
- Einmalige Antwort pro Anfrage

**Agent:**
- Plant Schritte zur Zielerreichung
- Nutzt Werkzeuge (Tools) für externe Aktionen
- Hat einen Zustand (Memory) über mehrere Schritte
- Führt einen Reasoning-Loop aus (Reason → Act → Observe)

## Teil 2 – ReAct-Pattern implementieren (25 min)
ReAct (Reason + Act + Observe) ist das Herzstück vieler Agenten. Wir implementieren den Loop von Hand.

**Pflichtschritte:**
- Definiere mindestens zwei Werkzeuge.
- Implementiere den Reasoning-Loop.
- Beobachte, wie der Agent seine Schritte plant.

**Soll-Ergebnis:** Ein funktionierender ReAct-Agent mit sichtbarem Reasoning.

In [ ]:
import ast
import json

# Wissensbasis für das Beispiel
KNOWLEDGE = {
    "python": "Python ist eine interpretierte, objektorientierte Programmiersprache.",
    "transformer": "Transformer sind eine Deep-Learning-Architektur für NLP, basierend auf Attention.",
    "rag": "RAG (Retrieval Augmented Generation) kombiniert Retrieval mit einem generativen Modell.",
}


def safe_eval_expr(expression):
    """Sichere Auswertung mathematischer Ausdrücke."""
    allowed_binary_ops = {
        ast.Add: lambda left, right: left + right,
        ast.Sub: lambda left, right: left - right,
        ast.Mult: lambda left, right: left * right,
        ast.Div: lambda left, right: left / right,
        ast.FloorDiv: lambda left, right: left // right,
        ast.Mod: lambda left, right: left % right,
        ast.Pow: lambda left, right: left ** right,
    }
    allowed_unary_ops = {
        ast.UAdd: lambda value: value,
        ast.USub: lambda value: -value,
    }

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_binary_ops:
            return allowed_binary_ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_unary_ops:
            return allowed_unary_ops[type(node.op)](_eval(node.operand))
        raise RuntimeError(f"Unsupported calculator expression: {expression}")

    syntax_tree = ast.parse(expression, mode="eval")
    return _eval(syntax_tree)


def tool_calculator(expression):
    """Tool 1: Rechner für mathematische Ausdrücke."""
    result = safe_eval_expr(expression)
    return f"Ergebnis: {result}"


def tool_lookup(topic):
    """Tool 2: Wissensbasis-Abfrage."""
    if topic not in KNOWLEDGE:
        available = ", ".join(KNOWLEDGE.keys())
        raise RuntimeError(f"Unbekanntes Thema: {topic}. Verfügbare: {available}")
    return KNOWLEDGE[topic]


print("Tool 1 (calculator): Berechnet mathematische Ausdrücke")
print("Tool 2 (lookup): Durchsucht die Wissensbasis")
print("\nTest:", tool_calculator("23 * 17"))
print("Test:", tool_lookup("rag"))

In [ ]:
class ReActAgent:
    """ReAct-Agent: Reason → Act → Observe Loop."""

    def __init__(self, model):
        self.model = model
        self.tools = {
            "calculator": tool_calculator,
            "lookup": tool_lookup,
        }
        self.memory = []
        self.max_steps = 5

    def system_prompt(self):
        return (
            "Du bist ein ReAct-Agent. Du denkst laut und nutzt Werkzeuge.\n\n"
            "VERFÜGBARE WERKZEUGE:\n"
            "- calculator(expression): Berechnet mathematische Ausdrücke. Beispiel: calculator('2+2')\n"
            "- lookup(topic): Durchsucht die Wissensbasis. Beispiel: lookup('python')\n\n"
            "REGELN:\n"
            "1. DENKE zuerst laut (THINK: ...)\n"
            "2. Wenn du ein Werkzeug brauchst, rufe es auf mit: ACTION: tool_name(arg)\n"
            "3. Nach jedem Tool-Aufruf beobachte das Ergebnis (OBSERVE: ...)\n"
            "4. Wenn du die Antwort weißt, antworte mit: FINAL: <deine Antwort>\n\n"
            "Beispiel:\n"
            "THINK: Ich muss 23 * 17 berechnen.\n"
            "ACTION: calculator('23 * 17')\n"
            "OBSERVE: Ergebnis: 391\n"
            "FINAL: Das Ergebnis von 23 * 17 ist 391."
        )

    def parse_response(self, text):
        """Parst die三种 Antworttypen: THINK, ACTION, FINAL."""
        think_match = re.search(r"THINK:\s*(.+?)(?=\nACTION:|\nFINAL:|$)", text, re.DOTALL)
        action_match = re.search(r"ACTION:\s*(\w+)\(([^)]+)\)", text)
        final_match = re.search(r"FINAL:\s*(.+)$", text, re.DOTALL)

        return {
            "think": think_match.group(1).strip() if think_match else None,
            "action": (action_match.group(1), action_match.group(2).strip()) if action_match else None,
            "final": final_match.group(1).strip() if final_match else None,
        }

    def run(self, query):
        """Führt den ReAct-Loop aus."""
        self.memory = [
            {"role": "system", "content": self.system_prompt()},
            {"role": "user", "content": query},
        ]

        print(f"Frage: {query}\n")

        for step in range(self.max_steps):
            response = ollama.chat(
                model=self.model,
                messages=self.memory,
                think=False,
                options={"temperature": 0.3, "num_predict": 200},
                keep_alive="10m",
            )["message"]["content"].strip()

            print(f"--- Schritt {step + 1} ---")
            print(response)
            print()

            self.memory.append({"role": "assistant", "content": response})

            parsed = self.parse_response(response)

            # Tool-Ausführung
            if parsed["action"]:
                tool_name, arg = parsed["action"]
                if tool_name not in self.tools:
                    raise RuntimeError(f"Unbekanntes Tool: {tool_name}")

                observation = self.tools[tool_name](arg)
                print(f"→ Tool-Ausgabe: {observation}\n")
                self.memory.append({"role": "user", "content": f"OBSERVE: {observation}"})
                continue

            # Finale Antwort
            if parsed["final"]:
                print(f"✓ Fertige Antwort: {parsed['final']}")
                return parsed["final"]

        raise RuntimeError("Agent hat die maximale Schrittanzahl überschritten.")


# Test des ReAct-Agenten
agent = ReActAgent(MODEL)
result = agent.run("Berechne 15 * 13 und erkläre mir dann, was RAG ist.")

## Teil 3 – Tool Use / Function Calling mit Ollama (20 min)
Ollama unterstützt Function Calling über das Tool-Interface. Wir nutzen die eingebaute Unterstützung.

**Pflichtschritte:**
- Definiere ein Tool-Schema im JSON-Format.
- Nutze Ollamas Tool-Calling-Feature.
- Parse und führe den Tool-Aufruf aus.

**Soll-Ergebnis:** Ein Agent, der Ollamas natives Function Calling nutzt.

In [ ]:
# Tool-Definitionen im JSON-Format für Ollama
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Berechnet mathematische Ausdrücke",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Der mathematische Ausdruck, z.B. '2+2' oder '15 * 13'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Durchsucht die Wissensbasis nach einem Thema",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "Das Thema, nach dem gesucht werden soll (python, transformer, rag)",
                    }
                },
                "required": ["topic"],
            },
        },
    },
]


def execute_function(name, args):
    """Führt den tatsächlichen Funktionsaufruf aus."""
    if name == "calculator":
        return tool_calculator(args["expression"])
    elif name == "search_knowledge":
        return tool_lookup(args["topic"])
    else:
        raise RuntimeError(f"Unknown function: {name}")


# Test: Function Calling mit Ollama
messages = [
    {"role": "user", "content": "Was ist das Ergebnis von 42 + 58?"},
]

response = ollama.chat(
    model=MODEL,
    messages=messages,
    tools=TOOLS,
    keep_alive="10m",
)

print("Ollama Response:")
print(response)

# Prüfen, ob ein Tool aufgerufen wurde
if response.get("message", {}).get("tool_calls"):
    for tool_call in response["message"]["tool_calls"]:
        func_name = tool_call["function"]["name"]
        func_args = tool_call["function"]["arguments"]
        print(f"\n→ Tool-Aufruf erkannt: {func_name}")
        print(f"   Argumente: {func_args}")
        result = execute_function(func_name, func_args)
        print(f"   Ergebnis: {result}")
else:
    print("\nKein Tool-Aufruf in der Antwort enthalten.")

## Teil 4 – MCP-Server erstellen (20 min)
MCP (Model Context Protocol) ist ein Standard für Agenten-Tool-Kommunikation. Wir erstellen einen minimalen Server.

**Pflichtschritte:**
- Starte einen HTTP-Server mit MCP-Endpunkten.
- Definiere mindestens zwei Tools am Server.
- Verbinde den Agenten mit dem MCP-Server.

**Soll-Ergebnis:** Ein laufender MCP-Server mit Tool-Endpunkten.

In [ ]:
# Minimaler MCP-Server mit Standard-Bibliothek
# Dieser Server implementiert das MCP-Protokoll (JSON-RPC 2.0)

from http.server import HTTPServer, BaseHTTPRequestHandler
import json
import threading

MCP_PORT = 8765

# Tool-Registrierung am Server
MCP_TOOLS = {
    "calculator": {
        "description": "Berechnet mathematische Ausdrücke",
        "inputSchema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Mathematischer Ausdruck"},
            },
            "required": ["expression"],
        },
    },
    "get_time": {
        "description": "Gibt die aktuelle Uhrzeit zurück",
        "inputSchema": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
}


def execute_mcp_tool(name, params):
    """Führt MCP-Tool-Aufrufe aus."""
    if name == "calculator":
        result = safe_eval_expr(params["expression"])
        return {"success": True, "result": str(result)}
    elif name == "get_time":
        from datetime import datetime
        return {"success": True, "result": datetime.now().strftime("%H:%M:%S")}
    else:
        return {"success": False, "error": f"Unknown tool: {name}"}


class MCPHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        content_length = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(content_length)
        request = json.loads(body.decode("utf-8"))

        method = request.get("method")
        request_id = request.get("id", 1)

        if method == "tools/list":
            # Tool-Liste zurückgeben
            response = {
                "jsonrpc": "2.0",
                "id": request_id,
                "result": {
                    "tools": [
                        {"name": name, **schema} for name, schema in MCP_TOOLS.items()
                    ]
                },
            }
        elif method == "tools/call":
            # Tool ausführen
            tool_name = request.get("params", {}).get("name")
            tool_args = request.get("params", {}).get("arguments", {})
            result = execute_mcp_tool(tool_name, tool_args)
            response = {
                "jsonrpc": "2.0",
                "id": request_id,
                "result": {
                    "content": [{"type": "text", "text": json.dumps(result)}],
                },
            }
        else:
            response = {
                "jsonrpc": "2.0",
                "id": request_id,
                "error": {"code": -32601, "message": f"Method not found: {method}"},
            }

        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(response).encode("utf-8"))

    def log_message(self, format, *args):
        pass  # Server-Logs unterdrücken


def start_mcp_server(port):
    server = HTTPServer(("127.0.0.1", port), MCPHandler)
    print(f"MCP-Server läuft auf http://127.0.0.1:{port}")
    return server


# Server im Hintergrund starten
mcp_server = start_mcp_server(MCP_PORT)
server_thread = threading.Thread(target=mcp_server.serve_forever, daemon=True)
server_thread.start()
print("MCP-Server bereit!")

In [ ]:
# Test: MCP-Server abfragen

# 1. Tool-Liste abrufen
list_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {},
}

response = requests.post(
    f"http://127.0.0.1:{MCP_PORT}",
    json=list_request,
    headers={"Content-Type": "application/json"},
)

print("=== MCP tools/list ===")
print(json.dumps(response.json(), indent=2))

# 2. Tool aufrufen (calculator)
call_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "calculator",
        "arguments": {"expression": "99 + 77"},
    },
}

response = requests.post(
    f"http://127.0.0.1:{MCP_PORT}",
    json=call_request,
    headers={"Content-Type": "application/json"},
)

print("\n=== MCP tools/call (calculator) ===")
print(json.dumps(response.json(), indent=2))

# 3. Tool aufrufen (get_time)
time_request = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "get_time",
        "arguments": {},
    },
}

response = requests.post(
    f"http://127.0.0.1:{MCP_PORT}",
    json=time_request,
    headers={"Content-Type": "application/json"},
)

print("\n=== MCP tools/call (get_time) ===")
print(json.dumps(response.json(), indent=2))

## Teil 5 – Agentische RAG (10 min)
Agentische RAG kombiniert Retrieval mit Agenten-Logik für mehrstufige Informationsbeschaffung.

**Kurzinfo:**
- Klassisches RAG: Eine Retrieval-Anfrage → eine Antwort
- Agentisches RAG: Der Agent entscheidet, ob und wie er检索ert, kann mehrere Quellen prüfen und Antworten kombinieren
- Beispiel: "Vergleiche die Python-Definition mit der Transformer-Definition" → Zwei检索-Anfragen → Zusammenführung

**Soll-Ergebnis:** Ein Verständnis, warum agentische RAG mächtiger ist als einfaches RAG.

In [ ]:
# Agentisches RAG: Der Agent entscheidet über检索-Strategie

def agentic_rag_query(query, knowledge_base, agent):
    """
    Agentisches RAG:
    1. Agent analysiert die Anfrage
    2. Agent entscheidet, welche Topics检索ert werden müssen
    3. Agent führt检索s aus
    4. Agent kombiniert die Ergebnisse zur finalen Antwort
    """
    # Schritt 1: Analyse
    analyze_prompt = (
        f"Analysiere diese Anfrage und sage mir, welche Themen aus der Wissensbasis relevant sind.\n"
        f"Anfrage: {query}\n"
        f"Verfügbare Themen: {', '.join(knowledge_base.keys())}\n"
        "Antworte im Format: RELEVANT: thema1, thema2"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": analyze_prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 60},
        keep_alive="10m",
    )
    analysis = response["message"]["content"]
    print(f"1. Analyse: {analysis}")

    # Schritt 2: Retrieval-Entscheidung
    relevant_topics = []
    for topic in knowledge_base.keys():
        if topic.lower() in analysis.lower():
            relevant_topics.append(topic)
    print(f"2. Relevante Topics: {relevant_topics}")

    # Schritt 3: Retrieval durchführen
    retrieved = []
    for topic in relevant_topics:
        content = knowledge_base[topic]
        retrieved.append(f"**{topic}**: {content}")
        print(f"   Retrieval: {topic} → {content}")

    # Schritt 4: Antwort generieren
    combine_prompt = (
        f"Anfrage: {query}\n\n"
        f"Relevante Informationen:\n" + "\n".join(retrieved) + "\n\n"
        "Generiere eine präzise Antwort."
    )
    final_response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": combine_prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 100},
        keep_alive="10m",
    )
    return final_response["message"]["content"].strip()


# Test: Agentische RAG
query = "Vergleiche die Python-Definition mit der Transformer-Definition."
result = agentic_rag_query(query, KNOWLEDGE, agent)

print(f"\n=== Finale Antwort ===")
print(result)

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Dokumentiere einen vollständigen ReAct-Loop mit Reasoning, Action und Observation.
2. Zeige, dass der MCP-Server auf Tool-Anfragen reagiert.
3. Erkläre in 3 Sätzen, warum agentische RAG mächtiger ist als klassisches RAG.

**Transferaufgaben:**
1. Erweitere den MCP-Server um ein drittes Tool (z.B. Wetter, Datum, Textanalyse).
2. Implementiere einen Agenten, der automatisch den MCP-Server abfragt.
3. Baue eine Sicherheitsprüfung ein, die bösartige Tool-Aufrufe verhindert.